# Stage 2 — Project trajectories onto the value axis + de-risk analyses (Colab A100)

This is the **GPU half** of Stage 2. It must run on a GPU because it reads raw
residual-stream activations (the OpenAI-compatible API used for *generation* cannot
return those).

**Before this notebook you must have:**
1. Run [`mini_agent_colab.ipynb`](https://colab.research.google.com/github/abdelmagid07/latent_failiure_prediction/blob/main/stage2/notebooks/mini_agent_colab.ipynb) (CPU) after [`serve_qwen_colab.ipynb`](https://colab.research.google.com/github/abdelmagid07/latent_failiure_prediction/blob/main/stage2/notebooks/serve_qwen_colab.ipynb) (A100).
2. Downloaded `normalized.zip` from the mini agent notebook.

Then upload here: the **proxy axis** (`value_axis_proxy.npy` + `axis_manifest_proxy.json`)
and **`normalized.zip`**.

**Outputs:** `derisk_report.json`, `final_step_separation.png`, `noise_by_token_type.png`,
`snr_by_position.csv` — the three Jonas deliverables.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable GPU: Runtime → Change runtime type → A100 GPU'
print(torch.cuda.get_device_name(0))

In [ ]:
# Clone + install both stages. Confirm the printed commit is newest on main.
!git clone https://github.com/abdelmagid07/latent_failiure_prediction.git
%cd latent_failiure_prediction
!git log --oneline -1
!pip install -q -e stage1 -e stage2

### Upload the frozen proxy axis
Select `value_axis_proxy.npy` and `axis_manifest_proxy.json` (downloaded from Stage 1).

In [ ]:
import shutil, os
from google.colab import files
os.makedirs('stage1/data', exist_ok=True)
up = files.upload()
for name in up:
    shutil.move(name, f'stage1/data/{name}')
print('Axis files in stage1/data:', os.listdir('stage1/data'))

### Upload normalized trajectories
Zip your local `stage2/data/normalized/` (the `*.json` files from ingest) into
`normalized.zip` and upload it here.

In [ ]:
import zipfile, os, glob
from google.colab import files
os.makedirs('stage2/data/normalized', exist_ok=True)
up = files.upload()  # select normalized.zip
zname = next(iter(up))
with zipfile.ZipFile(zname) as z:
    z.extractall('stage2/data/normalized_extracted')
# Flatten any nested dirs so all *.json land directly in stage2/data/normalized/
for p in glob.glob('stage2/data/normalized_extracted/**/*.json', recursive=True):
    shutil.move(p, f'stage2/data/normalized/{os.path.basename(p)}')
jsons = glob.glob('stage2/data/normalized/*.json')
print(f'{len(jsons)} normalized trajectories ready')
assert jsons, 'No trajectory JSON found — check the zip contents.'

In [ ]:
# Sanity check the labels BEFORE spending GPU time: you need both classes.
import json, glob
outcomes = [json.load(open(p)).get('outcome') for p in glob.glob('stage2/data/normalized/*.json')]
n_succ = sum(1 for o in outcomes if o == 1)
n_fail = sum(1 for o in outcomes if o == 0)
print(f'success={n_succ}  failure={n_fail}  total={len(outcomes)}')
if n_succ == 0 or n_fail == 0:
    print('WARNING: only one class present — success-vs-failure analysis will be degenerate.')

In [ ]:
# Project every step onto the PROXY axis (layer auto from config=21; 36 layers).
%cd /content/latent_failiure_prediction/stage2
!python -m stage2.extract.project_steps \
  --traj-dir data/normalized \
  --output data/projections.parquet \
  --axis-path ../stage1/data/value_axis_proxy.npy

In [ ]:
# Run the three de-risk analyses with proxy labeling for the Jonas deliverable.
!python -m stage2.analyze.run_derisk \
  --projections data/projections.parquet \
  --output-dir data/proxy_derisk_report \
  --proxy

In [ ]:
# Download the deliverables.
from google.colab import files
for p in [
    'data/proxy_derisk_report/derisk_report.json',
    'data/proxy_derisk_report/final_step_separation.png',
    'data/proxy_derisk_report/noise_by_token_type.png',
    'data/proxy_derisk_report/snr_by_position.csv',
]:
    files.download(p)